# ZAIC 2026 LifeMate — v2: guard mới + fine-tune LoRA

**Cách dùng:** GPU **A100** (L4 cũng được nhưng train chậm hơn khoảng 2 lần). Upload `lifemate_baseline.zip` **bản mới** vào `MyDrive/ZAIC/` (ghi đè bản cũ) → **Run all** → gửi lại `results_v2.zip`.

Thời gian ước tính trên A100: cài đặt 5', baseline v2 10', train LoRA 60–90', đánh giá 15'. Cell train in ETA sau vài phút.
Nếu Colab ngắt giữa chừng: adapter đã lưu vào `MyDrive/ZAIC/ft_fit_adapter` sau khi train xong; khi đó đặt `RUN_TRAIN=False` rồi chạy lại.

In [ ]:
# ==== CẤU HÌNH ====
MODEL     = "Qwen/Qwen3-4B-Instruct-2507"
BATCH     = 32
EPOCHS    = 1        # 1 epoch trên (mẫu gốc + 1 bản augment) = mỗi mẫu được thấy 2 lần
AUG       = 1
LR        = 1e-4
RUN_BASE  = True     # baseline v2 (prompt + guard mới) với model gốc
RUN_TRAIN = True
RUN_FT    = True
DATA_URL  = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR = "/content/drive/MyDrive/ZAIC"
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_v2"
FT_DIR    = "/content/ft_fit"

In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess
code_zip = os.path.join(DRIVE_DIR, "lifemate_baseline.zip")
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(code_zip).extractall(WORK)
assert os.path.exists(f"{WORK}/lifemate/train_lora.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA)

In [ ]:
%%capture pipout
!pip install -q vllm peft
!pip uninstall -y torchao  # Colab ships torchao 0.10; peft refuses < 0.16 and we do not use it

In [ ]:
print(pipout.stderr[-1500:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt

## 1. Sanity (oracle + guard tests) và dữ liệu train

In [ ]:
%cd {WORK}
!python -m lifemate.priors --root {DATA} --ids splits/train_fit.txt --out priors_fit.json
!python -m lifemate.run --root {DATA} --split train --backend oracle --oracle-mistakes --priors priors_fit.json --out runs/oracle --batch 16 2>/dev/null
!python scripts/score.py --root {DATA} --split train --pred runs/oracle/submission.jsonl --by-tag | tee {RES}/sanity.txt
!python tests/test_guards.py --root {DATA} | tail -1 | tee -a {RES}/sanity.txt
!python -m lifemate.sft_data --root {DATA} --ids splits/train_fit.txt --aug-copies {AUG} --out sft_fit.jsonl

In [ ]:
%cd {WORK}
import time, json
def evaluate(tag, model, sets=("val_unseen", "val_seen"), extra=""):
    for S in sets:
        t = time.time()
        !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {model} --dtype {DTYPE} --batch {BATCH} {extra} --out runs/{tag}_{S} 2>&1 | tail -1
        !python scripts/score.py --root {DATA} --split train --pred runs/{tag}_{S}/submission.jsonl --by-tag | tee {RES}/score_{tag}_{S}.txt
        !python -m lifemate.report --root {DATA} --split train --pred runs/{tag}_{S}/submission.jsonl --show 6 > {RES}/report_{tag}_{S}.txt
        !head -12 {RES}/report_{tag}_{S}.txt
        os.makedirs(f"{RES}/{tag}_{S}", exist_ok=True)
        for fn in ["submission.jsonl", "traces.jsonl"]:
            shutil.copy(f"runs/{tag}_{S}/{fn}", f"{RES}/{tag}_{S}/{fn}")
        print(tag, S, "took", round(time.time() - t), "s")

## 2. Baseline v2: model gốc + guard mới + priors

In [ ]:
if RUN_BASE:
    evaluate("base2", MODEL, extra="--priors priors_fit.json")

## 3. Train LoRA trên train_fit (957 mẫu + augment). Theo dõi `loss` và `eta`

In [ ]:
%cd {WORK}
if RUN_TRAIN:
    !python -m lifemate.train_lora --data sft_fit.jsonl --model {MODEL} --out {FT_DIR} --epochs {EPOCHS} --lr {LR} 2>&1 | grep -vi warn
    assert os.path.exists(f"{FT_DIR}/adapter"), "train lỗi: xem log phía trên"
    shutil.copy(f"{FT_DIR}/train_log.txt", f"{RES}/train_log.txt")
    shutil.copytree(f"{FT_DIR}/adapter", os.path.join(DRIVE_DIR, "ft_fit_adapter"), dirs_exist_ok=True)
    print("adapter saved to Drive")

In [ ]:
# nếu bỏ qua train (RUN_TRAIN=False) thì dựng lại model merged từ adapter trong Drive
if RUN_FT and not os.path.exists(f"{FT_DIR}/merged/config.json"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(base, os.path.join(DRIVE_DIR, "ft_fit_adapter")).merge_and_unload()
    m.save_pretrained(f"{FT_DIR}/merged"); AutoTokenizer.from_pretrained(MODEL).save_pretrained(f"{FT_DIR}/merged")
    del m, base; torch.cuda.empty_cache()

## 4. Đánh giá model đã fine-tune

In [ ]:
if RUN_FT:
    evaluate("ft", f"{FT_DIR}/merged", extra="--priors priors_fit.json")
    evaluate("ftnopri", f"{FT_DIR}/merged", sets=("val_unseen",))

In [ ]:
%cd {WORK}
if RUN_FT:
    !head -30 splits/val_unseen.txt > splits/lat30.txt
    !python -m lifemate.run --root {DATA} --split train --ids splits/lat30.txt --backend vllm --model {FT_DIR}/merged --dtype {DTYPE} --batch 1 --priors priors_fit.json --out runs/lat 2>&1 | tail -1
    secs = sorted(json.loads(l)["seconds"] for l in open("runs/lat/traces.jsonl"))
    s = f"GPU={GPU_NAME} ft batch=1 n={len(secs)} mean={sum(secs)/len(secs):.2f}s max={secs[-1]:.2f}s"
    print(s); open(f"{RES}/latency.txt", "w").write(s + "\n")

## 5. Đóng gói → gửi lại `results_v2.zip`

In [ ]:
open(f"{RES}/config.txt", "w").write(f"MODEL={MODEL}\nEPOCHS={EPOCHS}\nAUG={AUG}\nLR={LR}\nGPU={GPU_NAME}\nDTYPE={DTYPE}\n")
z = shutil.make_archive("/content/results_v2", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_v2.zip"))
print("saved", os.path.join(DRIVE_DIR, "results_v2.zip"), round(os.path.getsize(z)/1e6, 1), "MB")
from google.colab import files
files.download(z)